In [1]:
from rag_helper import RAGBase
from ingest import load_course_data, build_index

In [2]:
import os
from getpass import getpass
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
# This opens a secure input box to paste your key
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")
openai_client = OpenAI()

In [3]:
documents = load_course_data()
index = build_index(documents)

In [4]:
assistant = RAGBase(index, openai_client)

In [5]:
assistant.rag('How does the agentic loop keep calling the model until it stops?')

('It keeps calling the model in a `while True` loop.\n\nEach iteration:\n1. sends the full message history to the model,\n2. checks the response for any `function_call`,\n3. runs the tool and appends the tool output,\n4. sets `has_function_calls = True` if any tool was called.\n\nThen it stops only when a model response comes back with no function calls:\n\n```python\nif has_function_calls == False:\n    break\n```\n\nSo the loop continues until the model returns a final answer without asking for more tools.',
 ResponseUsage(input_tokens=7111, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=117, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=7228))

In [6]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [7]:
#print number of chunks
print(f'Number of chunks: {len(chunks)}')


Number of chunks: 295


In [8]:
chunk_index = build_index(chunks)
assistant.index = chunk_index

In [9]:
assistant.rag('How does the agentic loop keep calling the model until it stops?')

('The loop keeps calling the model in a `while True` loop and checks each response for `function_call` items.\n\n- If the model returns a function call, the code runs the tool, adds the tool result to `messages`, and continues.\n- If the model returns only a normal `message` and no function calls, `has_function_calls` stays `False`, so the loop `break`s.\n\nSo the stop condition is: **no function calls in the current turn**.',
 ResponseUsage(input_tokens=2294, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=102, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2396))

In [10]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [11]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return chunk_index.search(
        query,
        num_results=5,
        boost_dict={'question': 3.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'}
    )

In [12]:
agent_tools = Tools()
agent_tools.add_tool(search)

In [13]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [14]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [15]:
instructions = """
You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering.
"""


In [16]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [17]:
result = runner.loop(
    prompt='How does the agentic loop work, and how is it different from plain RAG?',
    callback=callback,
)

-> Response received


-> Response received


-> Response received
